In [1]:
import os 
os.getcwd()
os.chdir('/home/liujiajun/projects/Hap_networks/module/09-25')
from Trace import track_community_evolution
from TempSnap import IOManager
mcan_tables = IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/McAN_raw_results_2022-01-01_to_2025-07-23.h5')
graphs = IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/Temporal_graphs_2022-01-01_to_2025-07-23.h5')
communities= IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/Community_structures_2022-01-01_to_2025-07-23.h5')
backbone_tables = IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/Backbone_tables_2022-01-01_to_2025-07-23.h5')
backbone_networks = IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/Backbone_networks_2022-01-01_to_2025-07-23.h5')


In [3]:
# 遍历 graphs2（图索引从 0 开始），规范地输出每个图的索引、顶点数，以及顶点 'Date' 字段的最小/最大值。
for idx, g in enumerate(graphs):
    if g is None:
        print(f"Graph[{idx}]: None (图对象为空)")
        continue
    try:
        # 收集所有非空的 Date 属性
        dates = [v['Date'] for v in g.vs if 'Date' in v.attributes() and v['Date'] is not None]
        if not dates:
            print(f"Graph[{idx}]: vertices={len(g.vs)}; no valid 'Date' values")
            continue
        # 直接使用 min/max（可适配 datetime.date / datetime.datetime / 字符串 等类型）
        min_date = min(dates)
        max_date = max(dates)
        print(f"Graph[{idx}]: vertices={len(g.vs)}; min Date = {min_date}; max Date = {max_date}")
    except Exception as e:
        print(f"Graph[{idx}]: 错误 - {e}")


Graph[0]: None (图对象为空)
Graph[1]: None (图对象为空)
Graph[2]: None (图对象为空)
Graph[3]: None (图对象为空)
Graph[4]: None (图对象为空)
Graph[5]: vertices=12; min Date = 2022-01-01; max Date = 2022-05-19
Graph[6]: vertices=60; min Date = 2022-01-01; max Date = 2022-06-16
Graph[7]: vertices=193; min Date = 2022-01-01; max Date = 2022-07-15
Graph[8]: vertices=380; min Date = 2022-01-01; max Date = 2022-08-11
Graph[9]: vertices=434; min Date = 2022-01-01; max Date = 2022-09-07
Graph[10]: vertices=447; min Date = 2022-01-01; max Date = 2022-10-03
Graph[11]: vertices=450; min Date = 2022-01-01; max Date = 2022-10-28
Graph[12]: vertices=451; min Date = 2022-01-01; max Date = 2022-12-01
Graph[13]: vertices=452; min Date = 2022-01-01; max Date = 2022-12-28
Graph[14]: vertices=466; min Date = 2022-01-01; max Date = 2023-01-25
Graph[15]: vertices=471; min Date = 2022-01-01; max Date = 2023-02-13
Graph[16]: vertices=477; min Date = 2022-01-01; max Date = 2023-03-20
Graph[17]: vertices=496; min Date = 2022-01-01; max 

In [5]:
attributes = g.vs.attributes()
print("顶点的属性有:", attributes)


顶点的属性有: ['name', 'Date', 'Location', 'ID', 'Lineage', 'Clade', 'Ancestor_ID', 'wdks']


In [8]:
tracking_chains_mpox= track_community_evolution(
    partitions=communities,
    extended_graphs=graphs,
    label_of_interest='B.1',
    tracking_label='Lineage',
    recording_label='Lineage',
    start_date='2022-05-19',
    end_date='2022-09-07',
    time_interval=28,
    similarity_threshold=0.4,
    weight_attr="wdks",
    n_processes=4,output_path="/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data"
)


Processing graphs from index 5 to 9, corresponding to approx. date range: 2022-05-19 to 2022-09-07
Using GPU similarity calculator.
Starting tracking from 139 target communities...
Identified 145 communities in the evolution chain (139 targets + 6 tracked predecessors). Tracking took 17.49s.
Generated tracking DataFrame with 145 rows.
Identified 16 root nodes and 139 labeled nodes. Total 140 unique start points for chain building.
Building chains in parallel using 4 processes...
Constructed 140 raw chains containing the label.
Reduced to 52 unique chains after deduplication. Chain construction took 1.12s.
Tracking results saved to /home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/tracking_results_label_B.1_20220519_20220907.h5.
Total processing time: 19.35s


In [2]:
tracking_chains_mpox= IOManager.load_from_hdf5('/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/tracking_results_label_B.1_20220519_20220907.h5')


In [4]:
tracking_chains_mpox[27]


,Community ID,Contains Label,Similarity,Lineage Counts,Matched From Date,Matched From Community ID,Core Node,Mutations_of_core_node,Matched From Time Index,Alternative Matched From
Date,,,,,,,,,,
2022-06-16,0,True,0.000,"{""B.1"": 13, ""B.1.1"": 1, ""B.1.10"": 1, ""B.1.19"":...",<NA>,<NA>,B.1.7,1(Insertion:G->-T-----------------------------...,<NA>,[]
2022-07-15,0,True,0.999,"{""B.1"": 22, ""B.1.1"": 1, ""B.1.10"": 1, ""B.1.19"":...",2022-06-16,0,B.1.7,1(Insertion:G->-T-----------------------------...,1,[]
2022-08-11,2,True,0.999,"{""B.1"": 34, ""B.1.1"": 1, ""B.1.10"": 1, ""B.1.19"":...",2022-07-15,0,B.1,1(Insertion:G->-T-----------------------------...,2,[]
2022-09-07,2,True,1.000,"{""B.1"": 35, ""B.1.1"": 1, ""B.1.10"": 1, ""B.1.19"":...",2022-08-11,2,B.1,1(Insertion:G->-T-----------------------------...,3,[]


In [3]:

from typing import List, Dict, Optional, Set, Tuple
import numpy as np
import colorsys
import igraph as ig
import plotly.graph_objects as go
import pandas as pd
from collections import defaultdict, Counter
import json


# ============= 1. 辅助函数 =============

def load_font(font_path):
    """加载字体"""
    font_family = 'Times'
    if font_path:
        try:
            from matplotlib import font_manager
            font_manager.fontManager.addfont(font_path)
            font_family = font_manager.FontProperties(fname=font_path).get_name()
        except Exception as e:
            print(f"字体加载错误: {e}，使用默认字体")
    return font_family


def get_lineage_attr(graphs):
    """获取Lineage属性名"""
    return 'Lineage' if any('Lineage' in g.vs[0].attributes() 
                          for g in graphs if g and g.vs and len(g.vs) > 0) else None


def create_empty_figure(message, width, height, font_family='Times'):
    """创建空白图形"""
    fig = go.Figure()
    fig.update_layout(
        title=message,
        annotations=[dict(text=message, x=0.5, y=0.5, showarrow=False)],
        width=width, height=height,
        font=dict(family=font_family)
    )
    return fig


def get_clade_colors(clades):
    """生成clade颜色映射（集中管理颜色生成逻辑）"""
    colors = {}
    sorted_clades = sorted(clades)
    for i, clade in enumerate(sorted_clades):
        hue = i / max(1, len(sorted_clades))
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.7)
        colors[clade] = f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'
    return colors


def clip_to_ellipse(x, y, center_x, center_y, x_radius, y_radius, margin=0.98):
    """将点限制在椭圆内（统一边界检查逻辑）"""
    rel_x = (x - center_x) / (x_radius * margin)
    rel_y = (y - center_y) / (y_radius * margin)
    dist_sq = rel_x * rel_x + rel_y * rel_y
    
    if dist_sq > 1:
        scale = margin / np.sqrt(dist_sq)
        x = center_x + (x - center_x) * scale
        y = center_y + (y - center_y) * scale
    return x, y


# ============= 2. 日期和数据处理 =============

def extract_graph_timestamps(graphs):
    """提取每个图的时间戳"""
    timestamps = []
    for g in graphs:
        if g and g.vs and 'Date' in g.vs.attributes():
            dates = [pd.to_datetime(v['Date'], errors='coerce') 
                    for v in g.vs if 'Date' in v.attributes() and v['Date']]
            valid_dates = [d for d in dates if pd.notna(d)]
            timestamps.append(max(valid_dates) if valid_dates else pd.NaT)
        else:
            timestamps.append(pd.NaT)
    return timestamps


def create_date_mapping_optimized(graphs, all_dates_to_map, debug=False):
    """为日期集合创建到图索引的映射"""
    graph_timestamps = extract_graph_timestamps(graphs)
    date_to_graph_idx = {}
    
    for date in sorted([d for d in all_dates_to_map if pd.notna(d)]):
        # 找到最后一个时间戳 <= date 的图
        candidates = [i for i, ts in enumerate(graph_timestamps) if pd.notna(ts) and ts <= date]
        
        if candidates:
            best_idx = max(candidates)
        else:
            best_idx = next((i for i, ts in enumerate(graph_timestamps) if pd.notna(ts)), 0)
        
        date_to_graph_idx[date.strftime('%Y-%m-%d')] = best_idx
    
    if debug:
        print("\n日期到图索引的映射:")
        for d, i in sorted(date_to_graph_idx.items()):
            print(f"  日期 {d} -> 映射到图索引 {i}")
            
    return date_to_graph_idx


def parse_clade_counts(counts_str, debug=False):
    """解析clade计数字符串"""
    try:
        counts_str = str(counts_str).replace("'", "\"")
        if counts_str.startswith('{') and counts_str.endswith('}'):
            return json.loads(counts_str)
    except Exception as e:
        if debug:
            print(f"解析谱系信息出错: {e}")
    return {}


def filter_nodes_by_clade(nodes, expected_clades, debug=False, date_str=None, community_id=None):
    """根据clade过滤节点"""
    if not expected_clades:
        return nodes
    
    filtered_nodes = []
    for clade, count in expected_clades.items():
        clade_nodes = sorted([n for n in nodes if n['clade'] == clade], 
                           key=lambda x: x['wdks'], reverse=True)
        filtered_nodes.extend(clade_nodes[:int(count)])
        
        if len(clade_nodes) < count and debug:
            print(f"警告: 社区 {date_str}_{community_id} Clade {clade} 预期 {count} 个，但只找到 {len(clade_nodes)} 个")
    
    return filtered_nodes if filtered_nodes else nodes


def process_communities_optimized(chains, graphs, communities, date_to_graph_idx, clade_attr, debug=False):
    """处理社区和节点数据"""
    node_data = {}
    community_data = {}
    processed_communities = set()
    target_clade_counts_col = f"{clade_attr} Counts"
    
    # 收集所有需要处理的任务
    tasks = set()
    for chain_df in chains:
        if not chain_df.empty:
            for date_idx, row in chain_df.iterrows():
                tasks.add((date_idx.strftime('%Y-%m-%d'), int(row['Community ID'])))

    for date_str, community_id in sorted(tasks):
        key = (date_str, community_id)
        if key in processed_communities:
            continue
        processed_communities.add(key)

        graph_idx = date_to_graph_idx.get(date_str)
        if graph_idx is None or not (0 <= graph_idx < len(graphs) and graphs[graph_idx]):
            if debug:
                print(f"警告: 图索引 {graph_idx} 无效，跳过日期 {date_str}")
            continue
        
        graph = graphs[graph_idx]
        
        # 查找对应的行数据
        row = None
        for df in chains:
            try:
                row = df.loc[(df.index.strftime('%Y-%m-%d') == date_str) & 
                           (df['Community ID'] == community_id)].iloc[0]
                break
            except (IndexError, KeyError):
                continue
        if row is None:
            continue

        # 解析预期的clade信息
        expected_clades = {}
        if target_clade_counts_col in row and pd.notna(row[target_clade_counts_col]):
            expected_clades = parse_clade_counts(row[target_clade_counts_col], debug)

        # 获取社区节点
        community_nodes = []
        if 0 <= graph_idx < len(communities) and communities[graph_idx] and \
           0 <= community_id < len(communities[graph_idx]):
            community_nodes = communities[graph_idx][community_id]
        
        # 收集节点信息
        found_nodes = []
        max_wdks = -1.0
        for node_name in community_nodes:
            try:
                gn = graph.vs.find(name=node_name)
                c = str(gn[clade_attr]) if clade_attr in gn.attributes() and gn[clade_attr] else None
                w = float(gn['wdks']) if 'wdks' in gn.attributes() else 0.0
                found_nodes.append({'name': node_name, 'node': gn, 'clade': c, 'wdks': w})
                if w > max_wdks:
                    max_wdks = w
            except:
                pass

        # 过滤节点并保存数据
        filtered_nodes = filter_nodes_by_clade(found_nodes, expected_clades, debug, date_str, community_id)
        all_counts = {col: row[col] for col in row.index if 'Counts' in col}
        community_data[key] = {'counts': all_counts, 'max_wdks': max_wdks}
        
        for node_info in filtered_nodes:
            nid = f"{date_str}_{community_id}_{node_info['name']}"
            attrs = {
                'date': date_str,
                'community_id': community_id,
                'node_name': node_info['name'],
                'is_core': False,
                'original_graph_idx': graph_idx,
                'wdks': node_info['wdks']
            }
            attrs.update({a: node_info['node'][a] for a in node_info['node'].attributes() if a != 'name'})
            node_data[nid] = attrs
        
        # 标记核心节点
        if filtered_nodes:
            core_name = max(filtered_nodes, key=lambda x: x['wdks'])['name']
            core_id = f"{date_str}_{community_id}_{core_name}"
            if core_id in node_data:
                node_data[core_id]['is_core'] = True
    
    return node_data, community_data


# ============= 3. 图创建 =============

def create_graph(node_data, community_data, chains):
    """创建igraph图对象"""
    G = ig.Graph(directed=True)
    if not node_data:
        return G
        
    G.add_vertices(list(node_data.keys()))
    for v in G.vs:
        v.update_attributes(node_data.get(v['name'], {}))
    
    # 按社区组织节点
    nodes_by_community = defaultdict(list)
    for nid, attrs in node_data.items():
        nodes_by_community[(attrs['date'], attrs['community_id'])].append(nid)
    
    # 添加社区内边
    edge_list, attr_list = [], []
    for nodes in nodes_by_community.values():
        if len(nodes) > 1:
            for i in range(len(nodes)):
                for j in range(i + 1, len(nodes)):
                    edge_list.append((nodes[i], nodes[j]))
                    attr_list.append({'type': 'intra'})

    # 添加核心节点间的边
    core_nodes = {(attrs['date'], attrs['community_id']): nid 
                  for nid, attrs in node_data.items() if attrs.get('is_core')}
    
    for df in [d for d in chains if not d.empty]:
        df.index = pd.to_datetime(df.index)
        for i in range(len(df) - 1):
            src_dt = df.index[i].strftime('%Y-%m-%d')
            tgt_dt = df.index[i+1].strftime('%Y-%m-%d')
            src_cid = int(df.iloc[i]['Community ID'])
            tgt_cid = int(df.iloc[i+1]['Community ID'])
            
            src_core = core_nodes.get((src_dt, src_cid))
            tgt_core = core_nodes.get((tgt_dt, tgt_cid))
            
            if src_core and tgt_core:
                sim = float(df.iloc[i+1].get('Similarity', 0.5))
                evo = float(df.iloc[i+1].get('evo_weights', sim))
                edge_list.append((src_core, tgt_core))
                attr_list.append({'type': 'core', 'weight': sim, 'evo_weight': evo})

    if edge_list:
        G.add_edges(edge_list)
        for i, attrs in enumerate(attr_list):
            G.es[i].update_attributes(attrs)
            
    return G


# ============= 4. 节点分布策略（优化版）=============

def distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius, 
                                 radii, angles, pos):
    """通用的椭圆内节点分布函数"""
    for node_id, r, theta in zip(node_ids, radii, angles):
        x = center_x + x_radius * r * np.cos(theta)
        y = center_y + y_radius * r * np.sin(theta)
        x, y = clip_to_ellipse(x, y, center_x, center_y, x_radius, y_radius)
        pos[node_id] = (x, y)


def distribute_nodes_circular(node_ids, center_x, center_y, x_radius, y_radius, spread_factor, pos):
    """圆形分布（小节点数）"""
    n = len(node_ids)
    if n == 0:
        return
    
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    np.random.shuffle(angles)
    radii = np.random.uniform(0.8, 0.98, n) * spread_factor
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius, 
                                radii, angles, pos)


def distribute_nodes_with_jitter(node_ids, center_x, center_y, x_radius, y_radius, spread_factor, pos):
    """多环分布（大节点数）"""
    n = len(node_ids)
    if n == 0:
        return
    
    # 计算环数和每环节点数
    num_rings = max(1, int(np.ceil(np.sqrt(n) / 1.6)))
    ring_radii = np.linspace(0.15, 1.0, num_rings)
    ideal_weights = ring_radii
    counts = np.floor(ideal_weights / np.sum(ideal_weights) * n).astype(int)
    counts[-1] += n - np.sum(counts)  # 调整余数
    
    # 生成所有节点的坐标
    radii_list, angles_list = [], []
    for r_base, cnt in zip(ring_radii, counts):
        if cnt <= 0:
            continue
        angles = np.linspace(0, 2*np.pi, cnt, endpoint=False) + np.random.uniform(-0.3, 0.3, cnt)
        radii = r_base * np.random.uniform(0.95, 1.05, cnt) * spread_factor
        radii_list.extend(radii)
        angles_list.extend(angles)
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii_list[:n], angles_list[:n], pos)


def distribute_uniform_edge_bias(node_ids, center_x, center_y, x_radius, y_radius, 
                                  outer_density_bias, pos):
    """边缘偏置分布"""
    n = len(node_ids)
    if n == 0:
        return
    
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    np.random.shuffle(angles)
    
    # 生成径向分布
    outer_k = int(max(1, min(1, outer_density_bias) * n))
    radii = np.empty(n)
    radii[:outer_k] = np.random.uniform(0.92, 1.0, outer_k)
    
    if n > outer_k:
        inner_radii = np.random.uniform(0.25, 0.9, n - outer_k)
        inner_radii.sort()
        radii[outer_k:] = inner_radii
    
    perm = np.random.permutation(n)
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii[perm], angles[perm], pos)


def distribute_concentric_relaxed(node_ids, center_x, center_y, x_radius, y_radius,
                                   ring_compactness, min_separation, push_out_strength, pos):
    """同心圆松散分布"""
    n = len(node_ids)
    if n == 0:
        return
    
    # 计算环分布
    num_rings = max(1, int(round(np.sqrt(n) / max(0.3, 2 - ring_compactness))))
    radii_levels = np.linspace(0.22, 1.0, num_rings)
    weights = radii_levels
    counts = np.maximum(1, (weights / np.sum(weights) * n).astype(int))
    
    # 调整节点数
    diff = n - np.sum(counts)
    if diff > 0:
        counts[-1] += diff
    elif diff < 0:
        for _ in range(abs(diff)):
            counts[np.argmax(counts)] -= 1
    
    # 生成点
    radii_list, angles_list = [], []
    for r_level, cnt in zip(radii_levels, counts):
        angles = np.linspace(0, 2*np.pi, cnt, endpoint=False) + np.random.uniform(-0.4, 0.4, cnt)
        radii = np.clip(r_level * np.random.uniform(0.92, 1.05, cnt), 0.05, 1.0)
        
        # 应用推出强度
        if push_out_strength > 0:
            mask = radii < 0.55
            radii[mask] += push_out_strength * (1 - radii[mask])
            radii = np.clip(radii, 0.05, 1.0)
        
        radii_list.extend(radii)
        angles_list.extend(angles)
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii_list[:n], angles_list[:n], pos)


# ============= 5. 布局计算 =============

def calculate_layout(G, node_data, community_data, 
                     horizontal_spacing, vertical_spacing,
                     node_spread_factor, x_radius_scale, y_radius_scale,
                     node_size_factor, fig_width, fig_height,
                     distribution_mode='legacy',
                     outer_density_bias=0.65,
                     ring_compactness=0.85,
                     min_separation=0.045,
                     push_out_strength=0.15,
                     debug=False):
    """计算图布局"""
    # 组织节点和社区
    nodes_by_key = defaultdict(list)
    for node_id, attrs in node_data.items():
        key = (attrs['date'], attrs['community_id'])
        nodes_by_key[key].append(node_id)
    
    sorted_dates = sorted(set(k[0] for k in nodes_by_key.keys()))
    communities_by_date = defaultdict(list)
    for date, comm_id in nodes_by_key.keys():
        communities_by_date[date].append(comm_id)
    
    # 计算缩放因子
    canvas_scale_factor = min(fig_width, fig_height) / 800
    base_layout_scale = (0.5 + node_size_factor / 20) * canvas_scale_factor
    node_size_adjustment = 0.6 + (node_size_factor / 15)
    
    date_to_x = {date: i * horizontal_spacing for i, date in enumerate(sorted_dates)}
    pos = {}
    community_centers = {}
    
    for key, node_ids in nodes_by_key.items():
        date, community = key
        x_base = date_to_x[date]
        
        # 计算y偏移
        same_date_comms = communities_by_date[date]
        comm_idx = same_date_comms.index(community)
        total_comms = len(same_date_comms)
        y_offset = (comm_idx - (total_comms - 1) / 2) * vertical_spacing
        
        # 识别核心节点
        core_node = next((n for n in node_ids if node_data[n].get('is_core')), None)
        num_nodes = len(node_ids)
        
        # 计算社区大小
        max_wdks = community_data.get(key, {}).get('max_wdks', 0.1)
        size_scale = max(0.3, min(1.5, max_wdks * 3 * base_layout_scale))
        
        counts_info = community_data.get(key, {}).get('counts', {})
        max_count = max([int(val) for val in counts_info.values() if str(val).isdigit()], default=0)
        size_factor = max(0.3, min(1.5, max(num_nodes, max_count) / 8 * base_layout_scale))
        size_scale = max(size_scale, size_factor)
        
        if num_nodes > 1:
            # 多节点社区：计算椭圆并分布节点
            base_radius = size_scale * node_size_adjustment * 1.3
            x_radius = x_radius_scale * base_radius
            y_radius = y_radius_scale * base_radius * 1.1
            
            community_centers[key] = {
                'x': x_base, 'y': y_offset,
                'x_radius': x_radius * 1.1, 'y_radius': y_radius * 1.1,
                'max_wdks': max_wdks, 'node_count': num_nodes, 'counts': counts_info
            }
            
            working_nodes = node_ids.copy()
            if core_node:
                pos[core_node] = (x_base, y_offset)
                working_nodes.remove(core_node)
            
            # 选择分布策略
            if working_nodes:
                if distribution_mode == 'uniform_edge_bias':
                    distribute_uniform_edge_bias(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                outer_density_bias, pos)
                elif distribution_mode == 'concentric_relaxed':
                    distribute_concentric_relaxed(working_nodes, x_base, y_offset, x_radius, y_radius,
                                                 ring_compactness, min_separation, push_out_strength, pos)
                else:  # legacy
                    if len(working_nodes) <= 5:
                        distribute_nodes_circular(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                 node_spread_factor * 0.9, pos)
                    else:
                        distribute_nodes_with_jitter(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                    node_spread_factor, pos)
        else:
            # 单节点社区
            pos[node_ids[0]] = (x_base, y_offset)
            community_centers[key] = {
                'x': x_base, 'y': y_offset,
                'x_radius': x_radius_scale * 0.5 * size_scale * node_size_adjustment,
                'y_radius': y_radius_scale * 0.5 * size_scale * node_size_adjustment,
                'max_wdks': max_wdks, 'node_count': 1, 'counts': counts_info
            }
    
    return pos, community_centers

# ============= 6. 绘图元素 =============

def add_community_ellipses(fig, community_centers):
    """添加社区椭圆"""
    for (date, comm_id), center in community_centers.items():
        theta = np.linspace(0, 2*np.pi, 16)
        x = center['x'] + center['x_radius'] * np.cos(theta) * 0.9
        y = center['y'] + center['y_radius'] * np.sin(theta) * 0.9
        
        hover_info = [f"Date: {date}", f"Community: {comm_id}"]
        for col, val in center['counts'].items():
            hover_info.append(f"{col}: {val}")
        hover_info.append(f"Nodes: {center['node_count']}")
        
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines',
            line=dict(color='rgb(100, 100, 100)', width=0.8),opacity=0.3,
            fill='none', hoverinfo='text', text="<br>".join(hover_info),
            showlegend=False
        ))


def add_edges(fig, G, pos, debug=False):
    """添加边"""
    # 社区内边
    intra_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name']) 
                  for e in G.es if e['type'] != 'core' 
                  and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    for source, target in intra_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        fig.add_trace(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None], mode='lines',
            line=dict(width=0.8, color='rgb(100, 100, 100)', dash='dot'),opacity=0.5,
            hoverinfo='none', showlegend=False
        ))
    
    # 核心边
    core_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name'], e) 
                 for e in G.es if e['type'] == 'core' 
                 and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    for source, target, e in core_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        hover_text = []
        
        if 'evo_weight' in e.attributes() and e['evo_weight'] is not None:
            try:
                hover_text.append(f"Evolution Weight: {float(e['evo_weight']):.2f}")
            except:
                hover_text.append(f"Evolution Weight: {e['evo_weight']}")
        
        fig.add_trace(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None], mode='lines',
            line=dict(width=1.0, color='rgb(100, 100, 180)'),opacity=0.7,
            hoverinfo='text', text="<br>".join(hover_text) if hover_text else None,
            showlegend=False
        ))


def build_node_hover_text(v, clade_attr, lineage_attr, hover_attrs):
    """构建节点悬停文本"""
    v_attrs = v.attributes()
    hover_parts = []
    
    # 节点标识
    is_core = v['is_core'] if 'is_core' in v_attrs else False
    node_id_value = v['ID'] if 'ID' in v_attrs else (v['node_name'] if 'node_name' in v_attrs else v['name'])
    node_clade = v[clade_attr] if clade_attr and clade_attr in v_attrs else None
    node_lineage = v[lineage_attr] if lineage_attr and lineage_attr in v_attrs else None
    
    # 构建标签
    label = f"{node_id_value}"
    if node_clade:
        label += f" ({node_clade})"
        if node_lineage and node_lineage != node_clade:
            label += f", {node_lineage}"
    elif node_lineage:
        label += f" ({node_lineage})"
    
    hover_parts.append(f"{'Core Node' if is_core else 'Node'}: {label}")
    
    # 其他属性
    skip_attrs = {'id', 'date', 'community', 'wdks', 'name', 'node_name', clade_attr, lineage_attr}
    for attr_name in sorted(hover_attrs):
        if attr_name in v_attrs and attr_name.lower() not in skip_attrs and attr_name not in skip_attrs:
            hover_parts.append(f"{attr_name}: {v[attr_name]}")
    
    # WDKS、日期、社区
    if 'wdks' in v_attrs:
        try:
            hover_parts.append(f"WDKS: {v['wdks']:.4f}")
        except:
            pass
    
    hover_parts.append(f"Date: {v['date'] if 'date' in v_attrs else 'N/A'}")
    hover_parts.append(f"Community: {v['community_id'] if 'community_id' in v_attrs else 'N/A'}")
    
    return "<br>".join(hover_parts)


def add_nodes(fig, G, pos, node_size_factor, clade_attr, lineage_attr, hover_attrs, clade_colors):
    """添加节点"""
    node_x, node_y, node_sizes, node_colors, hover_texts, node_symbols = [], [], [], [], [], []
    default_color = 'rgb(128, 128, 128)'
    
    for v in G.vs:
        node_id = v['name']
        if node_id not in pos:
            continue
        
        x, y = pos[node_id]
        node_x.append(x)
        node_y.append(y)
        
        # 节点大小和属性
        v_attrs = v.attributes()
        is_core = v['is_core'] if 'is_core' in v_attrs else False
        size = 1.5
        if 'wdks' in v_attrs and v['wdks'] is not None:
            try:
                wdks_value = float(v['wdks'])
                size = min(5 + wdks_value * node_size_factor / 4, 12)
            except:
                pass
        
        node_sizes.append(size)
        node_symbols.append('triangle-up' if is_core else 'circle')
        
        # 节点颜色
        if clade_attr and clade_attr in v_attrs and str(v[clade_attr]) in clade_colors:
            node_colors.append(clade_colors[str(v[clade_attr])])
        else:
            node_colors.append(default_color)
        
        # 悬停文本
        hover_texts.append(build_node_hover_text(v, clade_attr, lineage_attr, hover_attrs))
    
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers',
        hoverinfo='text', text=hover_texts,
        marker=dict(
            color=node_colors, size=node_sizes, symbol=node_symbols,
            line=dict(width=0, color='rgb(0, 0, 0)'),opacity=1,
        ),
        showlegend=False,
        hoverlabel=dict(namelength=-1),
        hoveron='points+fills',
        hovertemplate='%{text}<extra></extra>'
    ))


def add_legend(fig, G, clade_attr, clade_colors, font_family):
    """添加图例"""
    # 核心连接示例
    if any(e['type'] == 'core' for e in G.es):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=1.2, color='rgb(100, 100, 180)'),opacity=1,
            name='Community Connection',
            legendgroup='edges', showlegend=True
        ))
    
    # Clade/Lineage 颜色
    all_clades = set(str(G.vs[i][clade_attr])
                     for i in range(len(G.vs))
                     if clade_attr in G.vs[i].attributes() and G.vs[i][clade_attr])
    
    for clade in sorted(all_clades):
        if clade in clade_colors:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(size=8, color=clade_colors[clade]),
                name=str(clade), showlegend=True, legendgroup='Lineages'
            ))


# ============= 7. 主可视化函数 =============

def visualize_evolution_chains(
    final_chains: List[pd.DataFrame], 
    graphs: List[ig.Graph],
    communities: List[List[List[str]]],
    output_file: Optional[str] = None,  # 支持 .html, .png, .pdf, .jpg, .svg 等格式
    title: str = "Community Evolution Network",
    # 布局参数
    node_size_factor: float = 15,
    horizontal_spacing: float = 0.75,
    vertical_spacing: float = 1.5,
    node_spread_factor: float = 0.9,
    x_radius_scale: float = 0.5,
    y_radius_scale: float = 1.2,
    distribution_mode: str = 'legacy',
    outer_density_bias: float = 0.65,
    ring_compactness: float = 0.85,
    min_separation: float = 0.045,
    push_out_strength: float = 0.15,
    random_seed: Optional[int] = None,
    # 尺寸和外观参数
    dpi: int = 100,
    font_path: Optional[str] = None,
    hover_attrs: Optional[Set[str]] = None,
    fig_width: int = 1600,
    fig_height: int = 1000,
    # 日期和标签参数
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    time_interval: int = 7,
    recording_label: str = "Clade",
    date_format: str = "%Y-%m-%d",
    max_date_ticks: Optional[int] = None,
    debug: bool = False
) -> go.Figure:
    """
    社区演化链可视化主函数
    
    参数:
        output_file: 输出文件路径。根据扩展名自动选择格式：
            - .html: 交互式HTML图形（默认）
            - .png, .jpg, .jpeg: 静态位图（需要安装 kaleido）
            - .pdf, .svg: 静态矢量图（需要安装 kaleido）
            - 其他或无扩展名: 自动保存为HTML格式
    
    返回:
        plotly.graph_objects.Figure 对象
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    font_family = load_font(font_path)
    if hover_attrs is None:
        hover_attrs = {'ID', 'Location'}

    # 1. 过滤链并收集日期
    all_dates_to_process = set()
    processed_chains = []
    
    try:
        start_ts = pd.to_datetime(start_date) if start_date else pd.Timestamp.min
        end_ts = pd.to_datetime(end_date) if end_date else pd.Timestamp.max
    except Exception as e:
        return create_empty_figure(f"日期格式错误: {e}", fig_width, fig_height, font_family)

    for df in final_chains:
        if not df.empty:
            df.index = pd.to_datetime(df.index, errors='coerce').dropna()
            filtered_df = df[(df.index >= start_ts) & (df.index <= end_ts)]
            if not filtered_df.empty:
                processed_chains.append(filtered_df)
                all_dates_to_process.update(filtered_df.index)

    if not processed_chains:
        return create_empty_figure("在指定日期范围内未找到任何演化链数据", fig_width, fig_height, font_family)

    print(f"将在 {start_ts.date()} 到 {end_ts.date()} 范围内处理 {len(all_dates_to_process)} 个独特日期。")
    
    # 2. 数据处理
    clade_attr = recording_label
    lineage_attr = get_lineage_attr(graphs)
    
    date_to_graph_idx = create_date_mapping_optimized(graphs, all_dates_to_process, debug)
    node_data, community_data = process_communities_optimized(
        processed_chains, graphs, communities, date_to_graph_idx, clade_attr, debug
    )
    
    if not node_data:
        return create_empty_figure("处理后无有效数据可供可视化", fig_width, fig_height, font_family)
    
    # 3. 创建图和布局
    G = create_graph(node_data, community_data, processed_chains)
    if not G.vs:
        return create_empty_figure("创建图后无节点", fig_width, fig_height, font_family)

    pos, community_centers = calculate_layout(
        G, node_data, community_data, horizontal_spacing, vertical_spacing, 
        node_spread_factor, x_radius_scale, y_radius_scale, node_size_factor, 
        fig_width, fig_height, distribution_mode, outer_density_bias, 
        ring_compactness, min_separation, push_out_strength, debug
    )
    
    # 已禁用：左右偏移会导致同一时间点的社区不在一列，看起来杂乱
    # apply_core_edge_weight_distance(G, node_data, pos, community_centers)
    
    # 4. 生成颜色映射
    all_clades = set(str(G.vs[i][clade_attr]) for i in range(len(G.vs)) 
                     if clade_attr in G.vs[i].attributes() and G.vs[i][clade_attr])
    clade_colors = get_clade_colors(all_clades)
    
    # 5. 创建图形
    fig = go.Figure()
    add_community_ellipses(fig, community_centers)
    add_edges(fig, G, pos, debug)
    add_nodes(fig, G, pos, node_size_factor, clade_attr, lineage_attr, hover_attrs, clade_colors)
    add_legend(fig, G, clade_attr, clade_colors, font_family)

    # 6. 生成时间轴
    date_positions = defaultdict(list)
    for (date, _), center in community_centers.items():
        date_positions[date].append(center['x'])
        
    all_tick_dates = sorted(date_positions.keys())
    
    if max_date_ticks and len(all_tick_dates) > max_date_ticks:
        indices = np.round(np.linspace(0, len(all_tick_dates) - 1, max_date_ticks)).astype(int)
        tick_dates = [all_tick_dates[i] for i in sorted(list(set(indices)))]
    else:
        tick_dates = all_tick_dates
        
    tick_x = [float(np.median(date_positions[d])) for d in tick_dates]
    tick_text = [pd.to_datetime(d).strftime(date_format) for d in tick_dates]
    
    # 7. 计算X轴范围
    x_range = None
    if community_centers:
        left_edges = [c['x'] - c.get('x_radius', 0) for c in community_centers.values()]
        right_edges = [c['x'] + c.get('x_radius', 0) for c in community_centers.values()]
        
        min_bound = min(left_edges)
        max_bound = max(right_edges)
        padding = (max_bound - min_bound) * 0.001 if max_bound > min_bound else 0.5
        x_range = [min_bound - padding, max_bound + padding]

    # 8. 更新布局
    fig.update_layout(
        title=dict(
            text=title,
            font=dict(family=font_family, size=24, color="black"),
            x=0.5, y=0.99, xanchor="center", yanchor="top"
        ),
        showlegend=True,
        hovermode='closest',
        margin=dict(b=80, l=10, r=10, t=40),
        xaxis=dict(
            title=dict(text='Time', standoff=10, font=dict(family=font_family, size=20)),
            showgrid=True, tickmode='array',
            tickvals=tick_x, ticktext=tick_text, tickangle=0,
            tickfont=dict(family=font_family, size=18),
            range=x_range
        ),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='white',
        font=dict(family=font_family),
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.15,
            x=0.5, xanchor="center",
            font=dict(family=font_family, size=16),
            itemsizing="constant", itemwidth=30, traceorder='normal'
        ),
        width=fig_width, height=fig_height
    )
    
    # 9. 保存图形（根据文件扩展名自动选择格式）
    if output_file:
        import os
        file_ext = os.path.splitext(output_file)[1].lower()
        
        if file_ext == '.html':
            fig.write_html(output_file)
            print(f"交互式HTML图形已保存到: {output_file}")
            
        elif file_ext in ['.png', '.jpg', '.jpeg', '.pdf', '.svg', '.webp']:
            # Plotly 的 write_image 会自动使用 kaleido 作为后端
            try:
                fig.write_image(output_file, width=fig_width, height=fig_height)
                print(f"静态图形已保存到: {output_file}")
            except (ImportError, ValueError) as e:
                print(f"错误: 需要安装 kaleido 才能导出静态图 ({file_ext})")
                print("请运行: pip install -U kaleido")
                html_file = os.path.splitext(output_file)[0] + '.html'
                fig.write_html(html_file)
                print(f"已改为保存HTML格式到: {html_file}")
            except Exception as e:
                print(f"保存失败: {e}")
                html_file = os.path.splitext(output_file)[0] + '.html'
                fig.write_html(html_file)
                print(f"已改为保存HTML格式到: {html_file}")
                
        else:
            # 未知格式，默认保存为HTML
            output_file = output_file + '.html' if not file_ext else os.path.splitext(output_file)[0] + '.html'
            fig.write_html(output_file)
            print(f"警告: 未识别的格式，已保存为HTML: {output_file}")

    return fig




In [5]:
fig = visualize_evolution_chains(
    final_chains=tracking_chains_mpox, start_date='2022-05-19',
    end_date='2022-08-11', time_interval=28, graphs=graphs, recording_label='Lineage',
    communities=communities, dpi=300, font_path='/home/liujiajun/projects/Hap_networks/TIMES.TTF',
    output_file='/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/test.svg',
    fig_width=210*5, fig_height=297*5, debug=True,
    node_size_factor=20,
    vertical_spacing=8,
    horizontal_spacing=2,
    node_spread_factor=0.9,
    x_radius_scale=0.08*2,
    y_radius_scale=0.3*3,
    title='The evolution path of the Mpox IIb B.1 lineage',
)

将在 2022-05-19 到 2022-08-11 范围内处理 4 个独特日期。

日期到图索引的映射:
  日期 2022-05-19 -> 映射到图索引 5
  日期 2022-06-16 -> 映射到图索引 6
  日期 2022-07-15 -> 映射到图索引 7
  日期 2022-08-11 -> 映射到图索引 8
静态图形已保存到: /home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/test.svg


In [6]:
import matplotlib.font_manager as font_manager
cfont_path = '/home/liujiajun/projects/Hap_networks/TIMES.TTF'
font_manager.fontManager.addfont(cfont_path)
cprop = font_manager.FontProperties(fname=cfont_path, weight='bold',size=20)


In [ ]:
from typing import List, Dict, Optional, Set, Tuple
import numpy as np
import colorsys
import igraph as ig
import plotly.graph_objects as go
import pandas as pd
from collections import defaultdict, Counter
import json


# ============= 1. 辅助函数 =============

def load_font(font_path):
    """加载字体"""
    font_family = 'Times'
    if font_path:
        try:
            from matplotlib import font_manager
            font_manager.fontManager.addfont(font_path)
            font_family = font_manager.FontProperties(fname=font_path).get_name()
        except Exception as e:
            print(f"字体加载错误: {e}，使用默认字体")
    return font_family


def get_lineage_attr(graphs):
    """获取Lineage属性名"""
    return 'Lineage' if any('Lineage' in g.vs[0].attributes() 
                          for g in graphs if g and g.vs and len(g.vs) > 0) else None


def create_empty_figure(message, width, height, font_family='Times'):
    """创建空白图形"""
    fig = go.Figure()
    fig.update_layout(
        title=message,
        annotations=[dict(text=message, x=0.5, y=0.5, showarrow=False)],
        width=width, height=height,
        font=dict(family=font_family)
    )
    return fig


def get_clade_colors(clades):
    """生成clade颜色映射（集中管理颜色生成逻辑）"""
    colors = {}
    sorted_clades = sorted(clades)
    for i, clade in enumerate(sorted_clades):
        hue = i / max(1, len(sorted_clades))
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.7)
        colors[clade] = f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'
    return colors


def clip_to_ellipse(x, y, center_x, center_y, x_radius, y_radius, margin=0.98):
    """将点限制在椭圆内（统一边界检查逻辑）"""
    rel_x = (x - center_x) / (x_radius * margin)
    rel_y = (y - center_y) / (y_radius * margin)
    dist_sq = rel_x * rel_x + rel_y * rel_y
    
    if dist_sq > 1:
        scale = margin / np.sqrt(dist_sq)
        x = center_x + (x - center_x) * scale
        y = center_y + (y - center_y) * scale
    return x, y


# ============= 2. 日期和数据处理 =============

def extract_graph_timestamps(graphs):
    """提取每个图的时间戳"""
    timestamps = []
    for g in graphs:
        if g and g.vs and 'Date' in g.vs.attributes():
            dates = [pd.to_datetime(v['Date'], errors='coerce') 
                    for v in g.vs if 'Date' in v.attributes() and v['Date']]
            valid_dates = [d for d in dates if pd.notna(d)]
            timestamps.append(max(valid_dates) if valid_dates else pd.NaT)
        else:
            timestamps.append(pd.NaT)
    return timestamps


def create_date_mapping_optimized(graphs, all_dates_to_map, debug=False):
    """为日期集合创建到图索引的映射"""
    graph_timestamps = extract_graph_timestamps(graphs)
    date_to_graph_idx = {}
    
    for date in sorted([d for d in all_dates_to_map if pd.notna(d)]):
        # 找到最后一个时间戳 <= date 的图
        candidates = [i for i, ts in enumerate(graph_timestamps) if pd.notna(ts) and ts <= date]
        
        if candidates:
            best_idx = max(candidates)
        else:
            best_idx = next((i for i, ts in enumerate(graph_timestamps) if pd.notna(ts)), 0)
        
        date_to_graph_idx[date.strftime('%Y-%m-%d')] = best_idx
    
    if debug:
        print("\n日期到图索引的映射:")
        for d, i in sorted(date_to_graph_idx.items()):
            print(f"  日期 {d} -> 映射到图索引 {i}")
            
    return date_to_graph_idx


def parse_clade_counts(counts_str, debug=False):
    """解析clade计数字符串"""
    try:
        counts_str = str(counts_str).replace("'", "\"")
        if counts_str.startswith('{') and counts_str.endswith('}'):
            return json.loads(counts_str)
    except Exception as e:
        if debug:
            print(f"解析谱系信息出错: {e}")
    return {}


def filter_nodes_by_clade(nodes, expected_clades, debug=False, date_str=None, community_id=None):
    """根据clade过滤节点"""
    if not expected_clades:
        return nodes
    
    filtered_nodes = []
    for clade, count in expected_clades.items():
        clade_nodes = sorted([n for n in nodes if n['clade'] == clade], 
                           key=lambda x: x['wdks'], reverse=True)
        filtered_nodes.extend(clade_nodes[:int(count)])
        
        if len(clade_nodes) < count and debug:
            print(f"警告: 社区 {date_str}_{community_id} Clade {clade} 预期 {count} 个，但只找到 {len(clade_nodes)} 个")
    
    return filtered_nodes if filtered_nodes else nodes


def process_communities_optimized(chains, graphs, communities, date_to_graph_idx, clade_attr, debug=False):
    """处理社区和节点数据"""
    node_data = {}
    community_data = {}
    processed_communities = set()
    target_clade_counts_col = f"{clade_attr} Counts"
    
    # 收集所有需要处理的任务
    tasks = set()
    for chain_df in chains:
        if not chain_df.empty:
            for date_idx, row in chain_df.iterrows():
                tasks.add((date_idx.strftime('%Y-%m-%d'), int(row['Community ID'])))

    for date_str, community_id in sorted(tasks):
        key = (date_str, community_id)
        if key in processed_communities:
            continue
        processed_communities.add(key)

        graph_idx = date_to_graph_idx.get(date_str)
        if graph_idx is None or not (0 <= graph_idx < len(graphs) and graphs[graph_idx]):
            if debug:
                print(f"警告: 图索引 {graph_idx} 无效，跳过日期 {date_str}")
            continue
        
        graph = graphs[graph_idx]
        
        # 查找对应的行数据
        row = None
        for df in chains:
            try:
                row = df.loc[(df.index.strftime('%Y-%m-%d') == date_str) & 
                           (df['Community ID'] == community_id)].iloc[0]
                break
            except (IndexError, KeyError):
                continue
        if row is None:
            continue

        # 解析预期的clade信息
        expected_clades = {}
        if target_clade_counts_col in row and pd.notna(row[target_clade_counts_col]):
            expected_clades = parse_clade_counts(row[target_clade_counts_col], debug)

        # 获取社区节点
        community_nodes = []
        if 0 <= graph_idx < len(communities) and communities[graph_idx] and \
           0 <= community_id < len(communities[graph_idx]):
            community_nodes = communities[graph_idx][community_id]
        
        # 收集节点信息
        found_nodes = []
        max_wdks = -1.0
        for node_name in community_nodes:
            try:
                gn = graph.vs.find(name=node_name)
                c = str(gn[clade_attr]) if clade_attr in gn.attributes() and gn[clade_attr] else None
                w = float(gn['wdks']) if 'wdks' in gn.attributes() else 0.0
                found_nodes.append({'name': node_name, 'node': gn, 'clade': c, 'wdks': w})
                if w > max_wdks:
                    max_wdks = w
            except:
                pass

        # 过滤节点并保存数据
        filtered_nodes = filter_nodes_by_clade(found_nodes, expected_clades, debug, date_str, community_id)
        all_counts = {col: row[col] for col in row.index if 'Counts' in col}
        community_data[key] = {'counts': all_counts, 'max_wdks': max_wdks}
        
        for node_info in filtered_nodes:
            nid = f"{date_str}_{community_id}_{node_info['name']}"
            attrs = {
                'date': date_str,
                'community_id': community_id,
                'node_name': node_info['name'],
                'is_core': False,
                'original_graph_idx': graph_idx,
                'wdks': node_info['wdks']
            }
            attrs.update({a: node_info['node'][a] for a in node_info['node'].attributes() if a != 'name'})
            node_data[nid] = attrs
        
        # 标记核心节点
        if filtered_nodes:
            core_name = max(filtered_nodes, key=lambda x: x['wdks'])['name']
            core_id = f"{date_str}_{community_id}_{core_name}"
            if core_id in node_data:
                node_data[core_id]['is_core'] = True
    
    return node_data, community_data


# ============= 3. 图创建 =============

def create_graph(node_data, community_data, chains):
    """创建igraph图对象"""
    G = ig.Graph(directed=True)
    if not node_data:
        return G
        
    G.add_vertices(list(node_data.keys()))
    for v in G.vs:
        v.update_attributes(node_data.get(v['name'], {}))
    
    # 按社区组织节点
    nodes_by_community = defaultdict(list)
    for nid, attrs in node_data.items():
        nodes_by_community[(attrs['date'], attrs['community_id'])].append(nid)
    
    # 添加社区内边
    edge_list, attr_list = [], []
    for nodes in nodes_by_community.values():
        if len(nodes) > 1:
            for i in range(len(nodes)):
                for j in range(i + 1, len(nodes)):
                    edge_list.append((nodes[i], nodes[j]))
                    attr_list.append({'type': 'intra'})

    # 添加核心节点间的边
    core_nodes = {(attrs['date'], attrs['community_id']): nid 
                  for nid, attrs in node_data.items() if attrs.get('is_core')}
    
    for df in [d for d in chains if not d.empty]:
        df.index = pd.to_datetime(df.index)
        for i in range(len(df) - 1):
            src_dt = df.index[i].strftime('%Y-%m-%d')
            tgt_dt = df.index[i+1].strftime('%Y-%m-%d')
            src_cid = int(df.iloc[i]['Community ID'])
            tgt_cid = int(df.iloc[i+1]['Community ID'])
            
            src_core = core_nodes.get((src_dt, src_cid))
            tgt_core = core_nodes.get((tgt_dt, tgt_cid))
            
            if src_core and tgt_core:
                sim = float(df.iloc[i+1].get('Similarity', 0.5))
                evo = float(df.iloc[i+1].get('evo_weights', sim))
                edge_list.append((src_core, tgt_core))
                attr_list.append({'type': 'core', 'weight': sim, 'evo_weight': evo})

    if edge_list:
        G.add_edges(edge_list)
        for i, attrs in enumerate(attr_list):
            G.es[i].update_attributes(attrs)
            
    return G


# ============= 4. 节点分布策略（优化版）=============

def distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius, 
                                 radii, angles, pos):
    """通用的椭圆内节点分布函数"""
    for node_id, r, theta in zip(node_ids, radii, angles):
        x = center_x + x_radius * r * np.cos(theta)
        y = center_y + y_radius * r * np.sin(theta)
        x, y = clip_to_ellipse(x, y, center_x, center_y, x_radius, y_radius)
        pos[node_id] = (x, y)


def distribute_nodes_circular(node_ids, center_x, center_y, x_radius, y_radius, spread_factor, pos):
    """圆形分布（小节点数）"""
    n = len(node_ids)
    if n == 0:
        return
    
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    np.random.shuffle(angles)
    radii = np.random.uniform(0.8, 0.98, n) * spread_factor
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius, 
                                radii, angles, pos)


def distribute_nodes_with_jitter(node_ids, center_x, center_y, x_radius, y_radius, spread_factor, pos):
    """多环分布（大节点数）"""
    n = len(node_ids)
    if n == 0:
        return
    
    # 计算环数和每环节点数
    num_rings = max(1, int(np.ceil(np.sqrt(n) / 1.6)))
    ring_radii = np.linspace(0.15, 1.0, num_rings)
    ideal_weights = ring_radii
    counts = np.floor(ideal_weights / np.sum(ideal_weights) * n).astype(int)
    counts[-1] += n - np.sum(counts)  # 调整余数
    
    # 生成所有节点的坐标
    radii_list, angles_list = [], []
    for r_base, cnt in zip(ring_radii, counts):
        if cnt <= 0:
            continue
        angles = np.linspace(0, 2*np.pi, cnt, endpoint=False) + np.random.uniform(-0.3, 0.3, cnt)
        radii = r_base * np.random.uniform(0.95, 1.05, cnt) * spread_factor
        radii_list.extend(radii)
        angles_list.extend(angles)
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii_list[:n], angles_list[:n], pos)


def distribute_uniform_edge_bias(node_ids, center_x, center_y, x_radius, y_radius, 
                                  outer_density_bias, pos):
    """边缘偏置分布"""
    n = len(node_ids)
    if n == 0:
        return
    
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    np.random.shuffle(angles)
    
    # 生成径向分布
    outer_k = int(max(1, min(1, outer_density_bias) * n))
    radii = np.empty(n)
    radii[:outer_k] = np.random.uniform(0.92, 1.0, outer_k)
    
    if n > outer_k:
        inner_radii = np.random.uniform(0.25, 0.9, n - outer_k)
        inner_radii.sort()
        radii[outer_k:] = inner_radii
    
    perm = np.random.permutation(n)
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii[perm], angles[perm], pos)


def distribute_concentric_relaxed(node_ids, center_x, center_y, x_radius, y_radius,
                                   ring_compactness, min_separation, push_out_strength, pos):
    """同心圆松散分布"""
    n = len(node_ids)
    if n == 0:
        return
    
    # 计算环分布
    num_rings = max(1, int(round(np.sqrt(n) / max(0.3, 2 - ring_compactness))))
    radii_levels = np.linspace(0.22, 1.0, num_rings)
    weights = radii_levels
    counts = np.maximum(1, (weights / np.sum(weights) * n).astype(int))
    
    # 调整节点数
    diff = n - np.sum(counts)
    if diff > 0:
        counts[-1] += diff
    elif diff < 0:
        for _ in range(abs(diff)):
            counts[np.argmax(counts)] -= 1
    
    # 生成点
    radii_list, angles_list = [], []
    for r_level, cnt in zip(radii_levels, counts):
        angles = np.linspace(0, 2*np.pi, cnt, endpoint=False) + np.random.uniform(-0.4, 0.4, cnt)
        radii = np.clip(r_level * np.random.uniform(0.92, 1.05, cnt), 0.05, 1.0)
        
        # 应用推出强度
        if push_out_strength > 0:
            mask = radii < 0.55
            radii[mask] += push_out_strength * (1 - radii[mask])
            radii = np.clip(radii, 0.05, 1.0)
        
        radii_list.extend(radii)
        angles_list.extend(angles)
    
    distribute_nodes_in_ellipse(node_ids, center_x, center_y, x_radius, y_radius,
                                radii_list[:n], angles_list[:n], pos)


# ============= 5. 布局计算 =============

def calculate_layout(G, node_data, community_data, 
                     horizontal_spacing, vertical_spacing,
                     node_spread_factor, x_radius_scale, y_radius_scale,
                     node_size_factor, fig_width, fig_height,
                     distribution_mode='legacy',
                     outer_density_bias=0.65,
                     ring_compactness=0.85,
                     min_separation=0.045,
                     push_out_strength=0.15,
                     debug=False):
    """计算图布局"""
    # 组织节点和社区
    nodes_by_key = defaultdict(list)
    for node_id, attrs in node_data.items():
        key = (attrs['date'], attrs['community_id'])
        nodes_by_key[key].append(node_id)
    
    sorted_dates = sorted(set(k[0] for k in nodes_by_key.keys()))
    communities_by_date = defaultdict(list)
    for date, comm_id in nodes_by_key.keys():
        communities_by_date[date].append(comm_id)
    
    # 计算缩放因子
    canvas_scale_factor = min(fig_width, fig_height) / 800
    base_layout_scale = (0.5 + node_size_factor / 20) * canvas_scale_factor
    node_size_adjustment = 0.6 + (node_size_factor / 15)
    
    date_to_x = {date: i * horizontal_spacing for i, date in enumerate(sorted_dates)}
    pos = {}
    community_centers = {}
    
    for key, node_ids in nodes_by_key.items():
        date, community = key
        x_base = date_to_x[date]
        
        # 计算y偏移
        same_date_comms = communities_by_date[date]
        comm_idx = same_date_comms.index(community)
        total_comms = len(same_date_comms)
        y_offset = (comm_idx - (total_comms - 1) / 2) * vertical_spacing
        
        # 识别核心节点
        core_node = next((n for n in node_ids if node_data[n].get('is_core')), None)
        num_nodes = len(node_ids)
        
        # 计算社区大小
        max_wdks = community_data.get(key, {}).get('max_wdks', 0.1)
        size_scale = max(0.3, min(1.5, max_wdks * 3 * base_layout_scale))
        
        counts_info = community_data.get(key, {}).get('counts', {})
        max_count = max([int(val) for val in counts_info.values() if str(val).isdigit()], default=0)
        size_factor = max(0.3, min(1.5, max(num_nodes, max_count) / 8 * base_layout_scale))
        size_scale = max(size_scale, size_factor)
        
        if num_nodes > 1:
            # 多节点社区：计算椭圆并分布节点
            base_radius = size_scale * node_size_adjustment * 1.3
            x_radius = x_radius_scale * base_radius
            y_radius = y_radius_scale * base_radius * 1.1
            
            community_centers[key] = {
                'x': x_base, 'y': y_offset,
                'x_radius': x_radius * 1.1, 'y_radius': y_radius * 1.1,
                'max_wdks': max_wdks, 'node_count': num_nodes, 'counts': counts_info
            }
            
            working_nodes = node_ids.copy()
            if core_node:
                pos[core_node] = (x_base, y_offset)
                working_nodes.remove(core_node)
            
            # 选择分布策略
            if working_nodes:
                if distribution_mode == 'uniform_edge_bias':
                    distribute_uniform_edge_bias(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                outer_density_bias, pos)
                elif distribution_mode == 'concentric_relaxed':
                    distribute_concentric_relaxed(working_nodes, x_base, y_offset, x_radius, y_radius,
                                                 ring_compactness, min_separation, push_out_strength, pos)
                else:  # legacy
                    if len(working_nodes) <= 5:
                        distribute_nodes_circular(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                 node_spread_factor * 0.9, pos)
                    else:
                        distribute_nodes_with_jitter(working_nodes, x_base, y_offset, x_radius, y_radius, 
                                                    node_spread_factor, pos)
        else:
            # 单节点社区
            pos[node_ids[0]] = (x_base, y_offset)
            community_centers[key] = {
                'x': x_base, 'y': y_offset,
                'x_radius': x_radius_scale * 0.5 * size_scale * node_size_adjustment,
                'y_radius': y_radius_scale * 0.5 * size_scale * node_size_adjustment,
                'max_wdks': max_wdks, 'node_count': 1, 'counts': counts_info
            }
    
    return pos, community_centers

# ============= 6. 绘图元素 =============

def add_community_ellipses(fig, community_centers):
    """添加社区椭圆"""
    for (date, comm_id), center in community_centers.items():
        theta = np.linspace(0, 2*np.pi, 16)
        x = center['x'] + center['x_radius'] * np.cos(theta) * 0.9
        y = center['y'] + center['y_radius'] * np.sin(theta) * 0.9
        
        hover_info = [f"Date: {date}", f"Community: {comm_id}"]
        for col, val in center['counts'].items():
            hover_info.append(f"{col}: {val}")
        hover_info.append(f"Nodes: {center['node_count']}")
        
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines',
            line=dict(color='rgba(100, 100, 100, 0.3)', width=0.8),
            fill='none', hoverinfo='text', text="<br>".join(hover_info),
            showlegend=False
        ))


def add_edges(fig, G, pos, debug=False):
    """添加边"""
    # 社区内边
    intra_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name']) 
                  for e in G.es if e['type'] != 'core' 
                  and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    for source, target in intra_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        fig.add_trace(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None], mode='lines',
            line=dict(width=0.8, color='rgba(100, 100, 100, 0.5)', dash='dot'),
            hoverinfo='none', showlegend=False
        ))
    
    # 核心边
    core_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name'], e) 
                 for e in G.es if e['type'] == 'core' 
                 and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    for source, target, e in core_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        hover_text = []
        
        if 'evo_weight' in e.attributes() and e['evo_weight'] is not None:
            try:
                hover_text.append(f"Evolution Weight: {float(e['evo_weight']):.2f}")
            except:
                hover_text.append(f"Evolution Weight: {e['evo_weight']}")
        
        fig.add_trace(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None], mode='lines',
            line=dict(width=1.0, color='rgb(100, 100, 180)', opacity=0.35),
            hoverinfo='text', text="<br>".join(hover_text) if hover_text else None,
            showlegend=False
        ))


def build_node_hover_text(v, clade_attr, lineage_attr, hover_attrs):
    """构建节点悬停文本"""
    v_attrs = v.attributes()
    hover_parts = []
    
    # 节点标识
    is_core = v['is_core'] if 'is_core' in v_attrs else False
    node_id_value = v['ID'] if 'ID' in v_attrs else (v['node_name'] if 'node_name' in v_attrs else v['name'])
    node_clade = v[clade_attr] if clade_attr and clade_attr in v_attrs else None
    node_lineage = v[lineage_attr] if lineage_attr and lineage_attr in v_attrs else None
    
    # 构建标签
    label = f"{node_id_value}"
    if node_clade:
        label += f" ({node_clade})"
        if node_lineage and node_lineage != node_clade:
            label += f", {node_lineage}"
    elif node_lineage:
        label += f" ({node_lineage})"
    
    hover_parts.append(f"{'Core Node' if is_core else 'Node'}: {label}")
    
    # 其他属性
    skip_attrs = {'id', 'date', 'community', 'wdks', 'name', 'node_name', clade_attr, lineage_attr}
    for attr_name in sorted(hover_attrs):
        if attr_name in v_attrs and attr_name.lower() not in skip_attrs and attr_name not in skip_attrs:
            hover_parts.append(f"{attr_name}: {v[attr_name]}")
    
    # WDKS、日期、社区
    if 'wdks' in v_attrs:
        try:
            hover_parts.append(f"WDKS: {v['wdks']:.4f}")
        except:
            pass
    
    hover_parts.append(f"Date: {v['date'] if 'date' in v_attrs else 'N/A'}")
    hover_parts.append(f"Community: {v['community_id'] if 'community_id' in v_attrs else 'N/A'}")
    
    return "<br>".join(hover_parts)


def add_nodes(fig, G, pos, node_size_factor, clade_attr, lineage_attr, hover_attrs, clade_colors):
    """添加节点"""
    node_x, node_y, node_sizes, node_colors, hover_texts, node_symbols = [], [], [], [], [], []
    default_color = 'rgb(128, 128, 128)'
    
    for v in G.vs:
        node_id = v['name']
        if node_id not in pos:
            continue
        
        x, y = pos[node_id]
        node_x.append(x)
        node_y.append(y)
        
        # 节点大小和属性
        v_attrs = v.attributes()
        is_core = v['is_core'] if 'is_core' in v_attrs else False
        size = 1.5
        if 'wdks' in v_attrs and v['wdks'] is not None:
            try:
                wdks_value = float(v['wdks'])
                size = min(5 + wdks_value * node_size_factor / 4, 12)
            except:
                pass
        
        node_sizes.append(size)
        node_symbols.append('triangle-up' if is_core else 'circle')
        
        # 节点颜色
        if clade_attr and clade_attr in v_attrs and str(v[clade_attr]) in clade_colors:
            node_colors.append(clade_colors[str(v[clade_attr])])
        else:
            node_colors.append(default_color)
        
        # 悬停文本
        hover_texts.append(build_node_hover_text(v, clade_attr, lineage_attr, hover_attrs))
    
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers',
        hoverinfo='text', text=hover_texts,
        marker=dict(
            color=node_colors, 
            size=node_sizes, 
            symbol=node_symbols,
            line=dict(width=0)  # 移除 color 和 opacity
        ),
        opacity=1.0,  # 移到这里，作为 Scatter 的参数
        showlegend=False,
        hoverlabel=dict(namelength=-1),
        hoveron='points+fills',
        hovertemplate='%{text}<extra></extra>'
    ))


def add_legend(fig, G, clade_attr, clade_colors, font_family):
    """添加图例"""
    # 核心连接示例
    if any(e['type'] == 'core' for e in G.es):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=1.2, color='rgba(100, 100, 180, 0.35)'),
            name='Community Connection',
            legendgroup='edges', showlegend=True
        ))
    
    # Clade/Lineage 颜色
    all_clades = set(str(G.vs[i][clade_attr])
                     for i in range(len(G.vs))
                     if clade_attr in G.vs[i].attributes() and G.vs[i][clade_attr])
    
    for clade in sorted(all_clades):
        if clade in clade_colors:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(size=8, color=clade_colors[clade]),
                name=str(clade), showlegend=True, legendgroup='Lineages'
            ))


# ============= 7. 主可视化函数 =============

def visualize_evolution_chains(
    final_chains: List[pd.DataFrame], 
    graphs: List[ig.Graph],
    communities: List[List[List[str]]],
    output_file: Optional[str] = None,
    title: str = "Community Evolution Network",
    # 布局参数
    node_size_factor: float = 15,
    horizontal_spacing: float = 0.75,
    vertical_spacing: float = 1.5,
    node_spread_factor: float = 0.9,
    x_radius_scale: float = 0.5,
    y_radius_scale: float = 1.2,
    distribution_mode: str = 'legacy',
    outer_density_bias: float = 0.65,
    ring_compactness: float = 0.85,
    min_separation: float = 0.045,
    push_out_strength: float = 0.15,
    random_seed: Optional[int] = None,
    # 尺寸和外观参数
    dpi: int = 100,
    font_path: Optional[str] = None,
    hover_attrs: Optional[Set[str]] = None,
    fig_width: int = 1600,
    fig_height: int = 1000,
    # 日期和标签参数
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    time_interval: int = 7,
    recording_label: str = "Clade",
    date_format: str = "%Y-%m-%d",
    max_date_ticks: Optional[int] = None,
    debug: bool = False
) -> go.Figure:
    """社区演化链可视化主函数"""
    if random_seed is not None:
        np.random.seed(random_seed)

    font_family = load_font(font_path)
    if hover_attrs is None:
        hover_attrs = {'ID', 'Location'}

    # 1. 过滤链并收集日期
    all_dates_to_process = set()
    processed_chains = []
    
    try:
        start_ts = pd.to_datetime(start_date) if start_date else pd.Timestamp.min
        end_ts = pd.to_datetime(end_date) if end_date else pd.Timestamp.max
    except Exception as e:
        return create_empty_figure(f"日期格式错误: {e}", fig_width, fig_height, font_family)

    for df in final_chains:
        if not df.empty:
            df.index = pd.to_datetime(df.index, errors='coerce').dropna()
            filtered_df = df[(df.index >= start_ts) & (df.index <= end_ts)]
            if not filtered_df.empty:
                processed_chains.append(filtered_df)
                all_dates_to_process.update(filtered_df.index)

    if not processed_chains:
        return create_empty_figure("在指定日期范围内未找到任何演化链数据", fig_width, fig_height, font_family)

    print(f"将在 {start_ts.date()} 到 {end_ts.date()} 范围内处理 {len(all_dates_to_process)} 个独特日期。")
    
    # 2. 数据处理
    clade_attr = recording_label
    lineage_attr = get_lineage_attr(graphs)
    
    date_to_graph_idx = create_date_mapping_optimized(graphs, all_dates_to_process, debug)
    node_data, community_data = process_communities_optimized(
        processed_chains, graphs, communities, date_to_graph_idx, clade_attr, debug
    )
    
    if not node_data:
        return create_empty_figure("处理后无有效数据可供可视化", fig_width, fig_height, font_family)
    
    # 3. 创建图和布局
    G = create_graph(node_data, community_data, processed_chains)
    if not G.vs:
        return create_empty_figure("创建图后无节点", fig_width, fig_height, font_family)

    pos, community_centers = calculate_layout(
        G, node_data, community_data, horizontal_spacing, vertical_spacing, 
        node_spread_factor, x_radius_scale, y_radius_scale, node_size_factor, 
        fig_width, fig_height, distribution_mode, outer_density_bias, 
        ring_compactness, min_separation, push_out_strength, debug
    )
    

    
    # 4. 生成颜色映射
    all_clades = set(str(G.vs[i][clade_attr]) for i in range(len(G.vs)) 
                     if clade_attr in G.vs[i].attributes() and G.vs[i][clade_attr])
    clade_colors = get_clade_colors(all_clades)
    
    # 5. 创建图形
    fig = go.Figure()
    add_community_ellipses(fig, community_centers)
    add_edges(fig, G, pos, debug)
    add_nodes(fig, G, pos, node_size_factor, clade_attr, lineage_attr, hover_attrs, clade_colors)
    add_legend(fig, G, clade_attr, clade_colors, font_family)

    # 6. 生成时间轴
    date_positions = defaultdict(list)
    for (date, _), center in community_centers.items():
        date_positions[date].append(center['x'])
        
    all_tick_dates = sorted(date_positions.keys())
    
    if max_date_ticks and len(all_tick_dates) > max_date_ticks:
        indices = np.round(np.linspace(0, len(all_tick_dates) - 1, max_date_ticks)).astype(int)
        tick_dates = [all_tick_dates[i] for i in sorted(list(set(indices)))]
    else:
        tick_dates = all_tick_dates
        
    tick_x = [float(np.median(date_positions[d])) for d in tick_dates]
    tick_text = [pd.to_datetime(d).strftime(date_format) for d in tick_dates]
    
    # 7. 计算X轴范围
    x_range = None
    if community_centers:
        left_edges = [c['x'] - c.get('x_radius', 0) for c in community_centers.values()]
        right_edges = [c['x'] + c.get('x_radius', 0) for c in community_centers.values()]
        
        min_bound = min(left_edges)
        max_bound = max(right_edges)
        padding = (max_bound - min_bound) * 0.001 if max_bound > min_bound else 0.5
        x_range = [min_bound - padding, max_bound + padding]

    # 8. 更新布局
    fig.update_layout(
        title=dict(
            text=title,
            font=dict(family=font_family, size=24, color="black"),
            x=0.5, y=0.99, xanchor="center", yanchor="top"
        ),
        showlegend=True,
        hovermode='closest',
        margin=dict(b=80, l=10, r=10, t=40),
        xaxis=dict(
            title=dict(text='Time', standoff=10, font=dict(family=font_family, size=20)),
            showgrid=True, tickmode='array',
            tickvals=tick_x, ticktext=tick_text, tickangle=0,
            tickfont=dict(family=font_family, size=18),
            range=x_range
        ),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='white',
        font=dict(family=font_family),
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.15,
            x=0.5, xanchor="center",
            font=dict(family=font_family, size=16),
            itemsizing="constant", itemwidth=30, traceorder='normal'
        ),
        width=fig_width, height=fig_height
    )
    
    if output_file:
        fig.write_html(output_file)
        print(f"图形已保存到: {output_file}")

    return fig

In [15]:
fig = visualize_evolution_chains(
    final_chains=tracking_chains_mpox, start_date='2022-05-19',
    end_date='2022-08-11', time_interval=28, graphs=graphs, recording_label='Lineage',
    communities=communities, dpi=100, font_path='/home/liujiajun/projects/Hap_networks/TIMES.TTF',
    output_file='/home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/newtest.html',
    fig_width=2100*0.5, fig_height=2970*0.5, debug=True,
    node_size_factor=30,
    vertical_spacing=9,
    horizontal_spacing=2,
    node_spread_factor=0.95,
    x_radius_scale=0.08*1.5,
    y_radius_scale=0.24*2.5,
    title='The evolution path of the Mpox IIb B.1 lineage',
)

将在 2022-05-19 到 2022-08-11 范围内处理 4 个独特日期。

日期到图索引的映射:
  日期 2022-05-19 -> 映射到图索引 5
  日期 2022-06-16 -> 映射到图索引 6
  日期 2022-07-15 -> 映射到图索引 7
  日期 2022-08-11 -> 映射到图索引 8
图形已保存到: /home/liujiajun/projects/Hap_networks/module/09-25/09-25Mpox/analysed_data/newtest.html


In [77]:
def save_figure_to_html(fig, output_file, fig_width, fig_height, dpi=100):
    """将图表保存为HTML文件，确保与Jupyter中的显示一致"""
    if not output_file:
        return
    
    # 创建cprop字体字典（如果cprop是FontProperties对象）
    if 'cprop' in globals():
        if hasattr(cprop, 'get_name'):
            cprop_font_dict = dict(family=cprop.get_name(), size=20)
        else:
            cprop_font_dict = cprop
    else:
        cprop_font_dict = dict(family='Arial', size=20)  # 默认字体
        
    # 确保图表尺寸固定 - 移除无效的responsive属性
    fig.update_layout(
        autosize=False,
        width=fig_width * dpi / 100,
        height=fig_height * dpi / 100,
        margin=dict(b=100),
        # 保持图例字体设置
        legend=dict(
            font=cprop_font_dict  # 使用字体字典而不是直接使用对象
        )
    )
    
    # 保存为HTML文件，包含所有必要配置
    fig.write_html(
        output_file,
        include_plotlyjs=True,
        include_mathjax=False,
        full_html=True,
        auto_open=False,
        config={
            'displayModeBar': True,
            'responsive': False,  # 正确位置: 在config中设置responsive
            'scrollZoom': True,
            'toImageButtonOptions': {
                'format': 'png',
                'filename': 'evolution_visualization',
                'height': fig_height * dpi / 100,
                'width': fig_width * dpi / 100,
                'scale': 1
            }
        }
    )
    print(f"已将图表保存至: {output_file}")

def visualize_evolution_chains(
    final_chains: List[pd.DataFrame], 
    graphs: List[ig.Graph],
    communities: List[List[List[str]]],
    output_file: Optional[str] = None,
    title: str = "Community Evolution Network",
    # 布局参数
    node_size_factor: float = 15,
    horizontal_spacing: float = 0.75,  # 时间点水平间距（原垂直间距）
    vertical_spacing: float = 1.5,     # 社区垂直间距（原水平间距）
    node_spread_factor: float = 0.9,   # 节点分散程度
    x_radius_scale: float = 0.5,       # 椭圆水平半径（原垂直）
    y_radius_scale: float = 1.2,
    edge_length_factor: float = 1.0,         # 椭圆垂直半径（原水平）  # 边宽度系数
    dpi: int = 300,                    # 图像DPI
    font_path: str = None,
    hover_attrs: Optional[Set[str]] = None,
    fig_width: int = 1600,             # 交换宽高以适应水平布局
    fig_height: int = 1000,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    time_interval: int = 7,
    recording_label: str = "Clade",
    debug: bool = False
) -> go.Figure:
    """可视化社区演化链 - 水平时间轴布局"""
    # 基础设置
    font_family = load_font(font_path)
    
    # 创建cprop字体字典（如果cprop是FontProperties对象）
    if 'cprop' in globals():
        if hasattr(cprop, 'get_name'):
            # 如果cprop是FontProperties对象，提取字体名称
            cprop_font_dict = dict(family=cprop.get_name(), size=20)
        else:
            # 如果cprop已经是字典或其他格式
            cprop_font_dict = cprop
    else:
        # 如果cprop未定义，使用默认字体
        cprop_font_dict = dict(family=font_family, size=20)
    if hover_attrs is None:
        hover_attrs = {'ID', 'Date', 'Community', 'wdks', 'Location'}
    
    # 提取日期范围和图选择
    start_idx, end_idx, date_range = extract_date_range(graphs, start_date, end_date, time_interval)
    if start_idx == -1 or end_idx == -1 or date_range is None:
        return create_empty_figure("未找到匹配的日期范围", fig_width, fig_height)
    
    # 使用选定范围的图和社区
    selected_graphs = graphs[start_idx:end_idx+1]
    selected_communities = communities[start_idx:end_idx+1]
    
    # 设置属性名称
    clade_attr = recording_label
    lineage_attr = get_lineage_attr(selected_graphs)
    
    # 创建日期到图索引的映射
    date_to_graph_idx, graph_date_ranges = create_date_mapping(selected_graphs, final_chains, debug)
    
    # 处理节点和社区数据
    node_data, community_data = process_communities(
        final_chains, selected_graphs, selected_communities, 
        date_to_graph_idx, clade_attr, debug
    )
    
    # 如果没有数据，返回空图
    if not node_data:
        return create_empty_figure("无可视化数据", fig_width, fig_height)
    
    # 创建图形数据结构
    G = create_graph(node_data, community_data, final_chains)
    
    # 计算布局位置
    pos, community_centers = calculate_layout(
        G, node_data, community_data,
        horizontal_spacing, vertical_spacing, 
        node_spread_factor, x_radius_scale, y_radius_scale,
        node_size_factor, fig_width, fig_height,
        edge_length_factor  # 传递新参数
    )
    
    # 创建可视化元素
    fig = go.Figure()
    
    # 添加可视化元素 - 改变顺序确保节点在最上层
    add_community_ellipses(fig, community_centers)
    add_edges(fig, G, pos, debug)
    add_nodes(fig, G, pos, node_size_factor, clade_attr, lineage_attr, hover_attrs)
    add_legend(fig, G, clade_attr, cprop_font_dict)
    
    # 更新布局 - 水平时间轴
    fig.update_layout(
        title=dict(
            text=title, 
            font=dict(family=font_family, size=24, color="black"),
            x=0.5,
            y=0.99,  # 降低标题位置（原为1.0或默认值）
            xanchor="center",
            yanchor="top"
        ),
        showlegend=True,
        hovermode='closest',
        margin=dict(b=5, l=7, r=5, t=7),  # 增加顶部边距
        # X轴改为时间轴
        xaxis=dict(
            title=dict(text='日期', standoff=5, font=dict(family=font_family,size=20)),
            tickmode='array',
            tickvals=[pos[node_id][0] for date in sorted(set(node_data[n]['date'] for n in node_data)) 
                     for node_id in [next(n for n in node_data if node_data[n]['date'] == date)]],
            ticktext=sorted(set(node_data[n]['date'] for n in node_data)),
            tickangle=0,
            tickfont=dict(family=font_family,size=20),
            showgrid=True,
            ticklabelposition="outside top"
        ),
        # Y轴不显示刻度
        yaxis=dict(
            showgrid=False, 
            zeroline=False, 
            showticklabels=False,
        ),
        plot_bgcolor='white',
        font=dict(family=font_family),  # 使用普通字体字典
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15,  
            xanchor="center",
            x=0.5,
            font=cprop_font_dict,  # 使用字体字典而不是直接使用对象
            itemsizing="constant",
            itemwidth=50,
            traceorder="normal",
            tracegroupgap=8
        ),
        width=fig_width * dpi / 100,
        height=fig_height * dpi / 100
    )
    
    return fig

def add_edges(fig, G, pos, debug=False):
    """添加边 - 简化版本，使边长更大"""
    # 直接分类收集边
    intra_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name']) 
                  for e in G.es if e['type'] != 'core' 
                  and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    core_edges = [(G.vs[e.source]['name'], G.vs[e.target]['name'], e) 
                 for e in G.es if e['type'] == 'core' 
                 and G.vs[e.source]['name'] in pos and G.vs[e.target]['name'] in pos]
    
    # 添加内部边
    for source, target in intra_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        
        fig.add_trace(go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            mode='lines',
            line=dict(width=0.8, color='rgba(100, 100, 100, 0.5)', dash='dot'),
            hoverinfo='none',
            showlegend=False
        ))
    
    # 添加核心边
    for source, target, e in core_edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        
        # 边宽度
        line_width = 1.0
        
        # 悬停文本
        hover_text = []
        if 'weight' in e.attributes() and e['weight'] is not None:
            try:
                hover_text.append(f"Similarity: {float(e['weight']):.2f}")
            except (ValueError, TypeError):
                hover_text.append(f"Similarity: {e['weight']}")
                
        if 'evo_weight' in e.attributes() and e['evo_weight'] is not None:
            try:
                hover_text.append(f"Evolution Weight: {float(e['evo_weight']):.2f}")
            except (ValueError, TypeError):
                hover_text.append(f"Evolution Weight: {e['evo_weight']}")
        
        fig.add_trace(go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            mode='lines',
            line=dict(width=line_width, color='rgba(100, 100, 180, 0.35)'), # <-- 颜色固定
            hoverinfo='text',
            text="<br>".join(hover_text) if hover_text else None,
            showlegend=False
        ))


def add_legend(fig, G, clade_attr, custom_font=None):
    """添加图例 - 使用自定义字体"""
    # 添加核心节点演化边图例 - 不再有箭头
    if any(e['type'] == 'core' for e in G.es):
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='lines',
            line=dict(width=1, color='rgba(100, 100, 180, 0.35)'),
            name='社区连接',
            legendgroup='edges'
        ))
    
    # 添加节点颜色图例
    all_clades = set(str(G.vs[i][clade_attr]) 
                    for i in range(len(G.vs)) 
                    if clade_attr in G.vs[i].attributes() and G.vs[i][clade_attr])
    
    for i, clade in enumerate(sorted(all_clades)):
        hue = i / max(1, len(all_clades))
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.7)
        color = f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'
        
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color=color),
            name=f'{clade_attr}: {clade}',
            showlegend=True,
            legendgroup='clades'
        ))

fig = visualize_evolution_chains(
    final_chains=tracking_chains_mpox, start_date='2022-05-28',
    end_date='2022-08-20', time_interval=14, graphs=graphs, recording_label='Clade',
    communities=communities, dpi=200, font_path='/home/liujiajun/projects/Hap_networks/songti.ttf',      
    output_file=None,  # 设为None，不要在visualize函数中保存
    fig_width=0.9*210 / 25.4*100, fig_height=0.9*297 / 25.4*100,debug=True,
    node_size_factor=40,           # 适当减小节点大小
    vertical_spacing=1.8,          # 增加垂直间距
    horizontal_spacing=1.2,        # 增加水平间距
    node_spread_factor=1.5,        # 减小节点分散系数，确保在椭圆内
    x_radius_scale=0.1*0.8,           # 更均衡的椭圆比例
    y_radius_scale=0.15*0.8,           # 更均衡的椭圆比例
    edge_length_factor=0.2,        # 弱化边长度与权重的关联
        title='猴痘IIb B.1谱系的进化路径', 
)

# 使用专门的保存函数
save_figure_to_html(fig, "猴痘IIb B.1谱系的进化路径.html", fig_width=0.9*210 / 25.4*100, fig_height=0.9*297 / 25.4*100, dpi=200)
# 显示或保存图表
fig.show()


Processing date range: 2022-05-19 to 2022-09-07

图索引和日期范围映射:
  图索引[0] -> 日期范围: 2022-01-01 至 2022-05-19
  图索引[1] -> 日期范围: 2022-01-01 至 2022-06-16
  图索引[2] -> 日期范围: 2022-01-01 至 2022-07-15
  图索引[3] -> 日期范围: 2022-01-01 至 2022-08-11
  图索引[4] -> 日期范围: 2022-01-01 至 2022-09-07
已将图表保存至: 猴痘IIb B.1谱系的进化路径.html
已将图表保存至: 猴痘IIb B.1谱系的进化路径.html
